Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/lucianommartins/lab-sabadao/blob/main/examples/notebooks/03_golden_set_capability_smoke_tests.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench golden set functional invariants and python code verification

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This notebook demonstrates how to verify the functional correctness of foundation models using the 16 deterministic Golden Set capability invariants in `gbench`. You will run automated smoke tests against a local Ollama serving engine, understand two-sided presence matching, inspect `python_exec` unit test execution, and verify multimodal Base64 image transport.

## Learning objectives

1. Configure Ollama to serve a quantized Google Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 8192 tokens.
2. Execute the complete Golden Set capability smoke test suite (`gbench --golden-only`).
3. Understand how two-sided presence assertions (`contains_all`, `refusal`) prevent false positive evaluation results.
4. Inspect `python_exec` unit test execution for canonical code generation tasks.
5. Execute targeted capability checks using `--golden-tasks`.
6. Perform a clean session shutdown to terminate background servers and reclaim hardware memory.

## Useful resources

* [lab-sabadao GitHub repository](https://github.com/lucianommartins/lab-sabadao)
* [Ollama documentation](https://github.com/ollama/ollama)
* [Unsloth Gemma 4 QAT GGUF checkpoints](https://huggingface.co/unsloth/gemma-4-E4B-it-qat-GGUF)

## 1. Environment setup and installation

We clone the `lab-sabadao` repository from GitHub, change directory into the project root (`%cd lab-sabadao`), and install the package in editable mode (`%pip install -e .`). This builds and links the `gbench` CLI executable without installing unnecessary development linters.

In [ ]:
import os, sys
from pathlib import Path

# Safe environment setup: Always normalize to top-level repository
if Path("/content").exists():
    %cd -q /content
    if not Path("/content/lab-sabadao").is_dir():
        !git clone https://github.com/lucianommartins/lab-sabadao.git
    %cd -q /content/lab-sabadao
else:
    if not Path("pyproject.toml").is_file() and not Path("gbench").is_dir():
        if not Path("lab-sabadao").is_dir():
            !git clone https://github.com/lucianommartins/lab-sabadao.git
        %cd lab-sabadao

%pip install -e . -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available Golden Set invariant capability tests
!gbench --list golden

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil

if not shutil.which("ollama"):
    print("Installing Ollama locally...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama binary already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Writing custom Modelfile for QAT model

We create an Ollama `Modelfile.qat` that configures our quantized Google Gemma 4 model (`hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:latest`) with explicit parameters:
* **`num_ctx 8192`**: Context window of 8192 tokens.
* **`SYSTEM prompt`**: System instruction defining Gemma 4 AI assistant capabilities.

In [ ]:
HF_MODEL_ID = "hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:UD-Q4_K_XL"
modelfile_content = f"""FROM {HF_MODEL_ID}
PARAMETER num_ctx 8192
SYSTEM "You are a helpful Gemma 4 AI assistant with reasoning, vision, and tool calling capabilities."
"""
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write(modelfile_content)
print("Created Modelfile.qat with valid Ollama parameters (num_ctx 8192, SYSTEM prompt).")

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) using `ollama create -f Modelfile.qat`. This pulls the GGUF weights from Hugging Face Hub if not already cached. We then run a quick generation test (`ollama run`) to verify that the model loads into hardware memory and generates tokens correctly.

In [ ]:
import requests

MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from Modelfile.qat...")
!ollama create {MODEL_TAG} -f Modelfile.qat

print("Running quick generation smoke test via Ollama API (cold load into GPU VRAM)...")
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_TAG, "prompt": "Reply with the single word: READY.", "stream": False},
    timeout=300,
)
print("Smoke test response:", resp.json().get("response", "").strip())

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in resp.json().get("data", [])]
print("Available REST models:", models)

## 7. Listing all Golden Set capability tasks

We can list all 16 available Golden invariant tasks using `!gbench --list golden` to inspect the available task IDs, capability categories, and assertion descriptions.

In [ ]:
!gbench --list golden

## 8. Executing the 16-invariant Golden Set test suite

We execute the full capability smoke test suite using `!gbench --golden-only`. This sends 16 mandatory prompt challenges across math, coding, JSON formatting, reasoning, and instruction following to verify that the target model (`gemma4-qat:4b`) achieves a clean pass score.

In [ ]:
!gbench --golden-only \
        --models gemma4-qat:4b \
        --golden-model-id gemma4-qat:4b \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_golden

## 9. Inspecting invariant test results

Every golden task evaluation returns strict two-sided presence matching (required terms present, forbidden terms absent) and structured tool validation to verify capability invariants without false positives.

In [ ]:
import json, glob, os
from pathlib import Path
import pandas as pd

# Load and inspect the latest golden set evaluation results
results_base = Path("./results_golden")
run_dirs = sorted([d for d in results_base.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True) if results_base.exists() else []

if run_dirs:
    latest_dir = run_dirs[0]
    summary_path = latest_dir / "summary.json"
    
    records = []
    if summary_path.exists():
        with open(summary_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for m in data.get("models", []):
            for t in m.get("task_results", []):
                records.append({
                    "task_id": t.get("task_id", t.get("id", "N/A")),
                    "category": t.get("category", "N/A"),
                    "verdict": "PASS" if t.get("passed") or t.get("status") == "passed" else "FAIL",
                    "details": str(t.get("details", t.get("reason", "-")))[:80],
                })
    
    if not records:
        for jf in sorted(latest_dir.glob("*.json")):
            try:
                with open(jf, "r", encoding="utf-8") as f:
                    data = json.load(f)
                tasks = data.get("task_results", data.get("tasks", []))
                for t in tasks:
                    records.append({
                        "task_id": t.get("task_id", t.get("id", "N/A")),
                        "category": t.get("category", "N/A"),
                        "verdict": "PASS" if t.get("passed") or t.get("status") == "passed" else "FAIL",
                        "details": str(t.get("details", t.get("reason", "-")))[:80],
                    })
            except Exception:
                pass

    if records:
        df = pd.DataFrame(records)
        passed_count = sum(1 for r in records if r["verdict"] == "PASS")
        total_count = len(records)
        print(f"Golden Set Score: {passed_count}/{total_count} passed ({passed_count/total_count*100:.1f}%)")
        display(df)
    else:
        print("No golden evaluation records found in:", latest_dir)
else:
    print("No golden results directory found.")

## 10. Session cleanup and server shutdown

We terminate background Ollama server processes and remove temporary Modelfiles.

In [ ]:
import subprocess, os

subprocess.run(["pkill", "-f", "ollama"], check=False)
if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")
print("Session cleanup complete.")